# UdaPlay AI Research Agent

## Part 1 - Offline RAG Pipeline

This notebook implements the first stage of the UdaPlay project: a local Retrieval-Augmented Generation (RAG) pipeline for video game data.

The pipeline will:

- Load the provided game JSON files
- Store them in a persistent ChromaDB vector database
- Generate embeddings for semantic search
- Retrieve relevant game information from the local knowledge base
- Create reusable retrieval components for the agent workflow

The source data is stored in the `games` folder. Each JSON file represents one game and will be added as a document in the Chroma collection.

Example game record:

```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```

### Setup

In [ ]:
# Compatibility setup for the Udacity workspace
import importlib.util
import sys

if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")

In [ ]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [ ]:
# The local .env file should contain:
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"
# OPENAI_BASE_URL="https://openai.vocareum.com/v1"

In [ ]:
# Load environment variables
load_dotenv()

assert os.getenv("OPENAI_API_KEY") is not None
assert os.getenv("CHROMA_OPENAI_API_KEY") is not None

### VectorDB Instance

In [ ]:
# Create a persistent ChromaDB client
chroma_client = chromadb.PersistentClient(path="chromadb")

### Collection and Embeddings

In [ ]:
# Configure the OpenAI embedding function
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("CHROMA_OPENAI_API_KEY"),
    api_base=os.getenv("OPENAI_BASE_URL"),
    model_name="text-embedding-3-small"
)

In [ ]:
# Create or load the UdaPlay collection
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

### Add Documents

In [ ]:
# Load game records from the local games directory
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)

    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # Format each game as searchable text
    content = (
        f"Game: {game['Name']}. "
        f"Platform: {game['Platform']}. "
        f"Genre: {game['Genre']}. "
        f"Publisher: {game['Publisher']}. "
        f"Year of release: {game['YearOfRelease']}. "
        f"Description: {game['Description']}"
    )

    # Use the JSON filename as the unique document ID
    doc_id = os.path.splitext(file_name)[0]

    collection.upsert(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

In [ ]:
# Confirm that the game records were indexed
print(f"Games indexed: {collection.count()}")

### Semantic Search Demonstration

The following query demonstrates semantic retrieval from the persistent ChromaDB collection.

In [ ]:
query = "Which game is a realistic racing simulator?"

results = collection.query(
    query_texts=[query],
    n_results=3
)

print(f"Query: {query}\n")

for i, metadata in enumerate(results["metadatas"][0], start=1):
    print(f"Result {i}: {metadata['Name']}")
    print(f"Platform: {metadata['Platform']}")
    print(f"Year: {metadata['YearOfRelease']}")
    print(f"Description: {metadata['Description']}\n")